# Are the remaining false alarms water? A check against Sentinel-2

The best detector of [notebook 04](04_false_alarms_s1.ipynb) still has a FAR of 0.74 on independent labels. Most of those pixels got darker between the two Sentinel-1 passes, on flat ground ([notebook 05](05_terrain_s1.ipynb)), far from the mapped flood. Either the detector is wrong, or the water is real and the EMS maps do not show it.

This notebook looks at EMSR664, which holds 111 of the 130 km² of independent labels, in a third image: the Sentinel-2 pass of **23 May 2023, 10:06 UTC**, about 29 hours after the Sentinel-1 pass, with 2–9 % cloud on its tiles. EMSR762, the next largest, is too cloudy on the nearby passes to test.

**The test is one-sided.** Water seen by Sentinel-2 confirms the radar. Water *not* seen proves little: it may have receded in those 29 hours, or lie under vegetation. So every number is compared with what Sentinel-2 sees under the mapped flood itself, under ordinary dry land, and under surfaces dark in both radar passes.

Cost: about **20 PU** once (six images at 40 m, three 16-bit bands), checked against a budget before anything is sent. The radar images come from notebook 02.

In [ ]:
import csv
import os
from io import BytesIO
from pathlib import Path

import numpy as np

ACTIVATIONS = [
    "EMSR659",
    "EMSR664",
    "EMSR705",
    "EMSR762",
    "EMSR795",
    "EMSR800",
    "EMSR843",
    "EMSR855",
]
TEST = "EMSR664"
RESOLUTION_M = 40
CHIP = 128
SEED = 0
BUDGET_PU = float(os.environ.get("NOTEBOOK_BUDGET_PU", 25))  # nothing is sent above it
OPTICAL_FROM, OPTICAL_TO = "2023-05-23T09:50:00Z", "2023-05-23T10:20:00Z"

DATA_DIR = Path(os.environ.get("COPERNICUS_EMS_DATA_DIR", "~/.cache/copernicus-ems")).expanduser()
DATASET = DATA_DIR / "datasets" / f"italy-floods-s1-change-{RESOLUTION_M}m"
OPTICAL_DIR = DATA_DIR / TEST / "sentinel-2"
FIGURE = Path(os.environ.get("NOTEBOOK_FIGURE", "../docs/images/notebook-optical-s2.png"))
print("data:", DATA_DIR, "| dataset:", DATASET)

## 1. The five-band dataset of notebook 02

The dataset is rebuilt from the pairs in the data directory, locally and for free. Its train/val/test column is ignored: the folds are made by activation below.

In [ ]:
from copernicus_ems.dataset import build_dataset, find_pairs

pairs = find_pairs(DATA_DIR, ACTIVATIONS)
assert any(p.reference is not None for p in pairs), (
    "no pair with a reference: run notebook 02 first"
)
summary = build_dataset(pairs, DATASET, size=CHIP, min_observed=0.5, seed=SEED, reference=True)
rows = list(csv.DictReader((DATASET / "index.csv").open()))
made_from = {
    (p.activation, p.product): sorted({i["sensor"] for i in p.meta.get("made_from", [])})
    for p in pairs
}
for r in rows:
    r["made_from"] = ", ".join(made_from.get((r["activation"], r["product"]), []))
    r["same_image"] = "Sentinel-1" in r["made_from"]
FOLDS = sorted({r["activation"] for r in rows})
print(
    f"{summary.pairs} pairs -> {summary.chips} chips of {CHIP} px; {len(FOLDS)} folds: {', '.join(FOLDS)}"
)

## 2. Features

The eleven features of notebook 02: VV and VH in dB, their ratio, local mean and spread of VV, the pre-event VV and VH, and the changes ΔVV, ΔVH, raw and smoothed over 5×5 pixels.

In [ ]:
import rasterio
from scipy.ndimage import uniform_filter

from copernicus_ems.ml import FLOODED, IGNORE, LabelRaster

SINGLE = slice(0, 5)
NAMES = [
    "VV",
    "VH",
    "VV-VH",
    "VV mean",
    "VV spread",
    "VV pre",
    "VH pre",
    "dVV",
    "dVH",
    "dVV mean",
    "dVH mean",
]


def db(x):
    return 10 * np.log10(np.maximum(x, 1e-6))


def features(path):
    with rasterio.open(path) as src:
        vv, vh, vv_pre, vh_pre, _mask = src.read()
    vv_db, vh_db, vvp_db, vhp_db = db(vv), db(vh), db(vv_pre), db(vh_pre)
    mean = uniform_filter(vv_db, 5)
    spread = np.sqrt(np.maximum(uniform_filter(vv_db**2, 5) - mean**2, 0))
    dvv, dvh = vv_db - vvp_db, vh_db - vhp_db
    return np.stack(
        [
            vv_db,
            vh_db,
            vv_db - vh_db,
            mean,
            spread,
            vvp_db,
            vhp_db,
            dvv,
            dvh,
            uniform_filter(dvv, 5),
            uniform_filter(dvh, 5),
        ],
        axis=-1,
    ).astype(np.float32)


X = np.stack([features(DATASET / r["image"]) for r in rows])
LABELS = np.stack([LabelRaster.from_geotiff(DATASET / r["label"]).labels for r in rows])
Y = [LabelRaster.from_geotiff(DATASET / r["label"]) for r in rows]
km2 = (RESOLUTION_M / 1000) ** 2
print(
    f"{'activation':10} {'chips':>5} {'flooded km²':>12} {'independent km²':>16} {'same image km²':>15}"
)
for a in FOLDS:
    idx = [i for i, r in enumerate(rows) if r["activation"] == a]
    wet = [int((LABELS[i] == FLOODED).sum()) for i in idx]
    ind = sum(w for i, w in zip(idx, wet, strict=True) if not rows[i]["same_image"])
    print(
        f"{a:10} {len(idx):5d} {sum(wet) * km2:12.1f} {ind * km2:16.1f} {(sum(wet) - ind) * km2:15.1f}"
    )

## 3. The detector, tuned without EMSR664

The rule of notebook 04, VV < *t* dB, VV_pre ≥ *t*ₚ dB, patches of at least *n* pixels, with all three chosen on the other activations only.

In [ ]:
from scipy.ndimage import label as connected

from copernicus_ems.ml import Scores, evaluate

OBSERVED = LABELS != IGNORE
WET = LABELS == FLOODED
EIGHT = np.ones((3, 3), dtype=bool)
VV, PRE = X[..., 0], X[..., 5]


def csi(chips, predicted):
    hits = int((predicted & WET[chips]).sum())
    false = int((predicted & OBSERVED[chips] & ~WET[chips]).sum())
    return hits / max(int(WET[chips].sum()) + false, 1)


def min_size(predicted, n):
    if n <= 1:
        return predicted
    labelled, _ = connected(predicted, structure=EIGHT)
    big = np.bincount(labelled.ravel()) >= n
    big[0] = False
    return big[labelled]


train = np.flatnonzero(np.array([r["activation"] != TEST for r in rows]))
test = np.flatnonzero(np.array([r["activation"] == TEST for r in rows]))
t, tp = max(
    [(a, b) for a in np.arange(-22, -10, 0.5) for b in np.arange(-22, -8, 1.0)],
    key=lambda c: csi(train, (VV[train] < c[0]) & (PRE[train] >= c[1])),
)
base = (VV[train] < t) & (PRE[train] >= tp)
n = max((1, 2, 4, 8, 16, 32), key=lambda k: csi(train, np.stack([min_size(b, k) for b in base])))
PREDICTED = np.zeros_like(WET)
for i in test:
    PREDICTED[i] = min_size((VV[i] < t) & (PRE[i] >= tp), n)
scores = sum((evaluate(PREDICTED[i], Y[i]) for i in test), Scores.zero())
print(f"VV < {t:.1f} dB, VV_pre >= {tp:.0f} dB, patches >= {n} px")
print(
    f"{TEST}: CSI {scores.csi:.3f}  POD {scores.pod:.3f}  FAR {scores.far:.3f}  bias {scores.bias:.2f}"
)

## 4. The Sentinel-2 pass, on the grids of the radar images

Green (B03), near infrared (B08) and the scene classification (SCL) on exactly the grid of each radar image, as 16-bit integers. The cost is computed and checked before the first request; images already in the data directory cost nothing.

In [ ]:
from copernicus_ems.sentinelhub import SentinelHub, estimate_pu

EVALSCRIPT = """//VERSION=3
function setup() {
  return {input: ["B03", "B08", "SCL", "dataMask"],
          output: {id: "default", bands: 4, sampleType: "UINT16"}};
}
function evaluatePixel(s) { return [s.B03 * 10000, s.B08 * 10000, s.SCL, s.dataMask]; }
"""

radar = {}  # radar image -> its pairs
used = {r["product"] for r in rows if r["activation"] == TEST}  # only pairs with chips
for p in pairs:
    if p.activation == TEST and p.product in used:
        radar.setdefault(p.image, []).append(p)


def optical_path(image):
    return OPTICAL_DIR / f"S2_20230523T1006_{image.stem}.tif"


def optical_request(src):
    left, bottom, right, top = src.bounds
    return {
        "input": {
            "bounds": {
                "bbox": [left, bottom, right, top],
                "properties": {"crs": f"http://www.opengis.net/def/crs/EPSG/0/{src.crs.to_epsg()}"},
            },
            "data": [
                {
                    "type": "sentinel-2-l2a",
                    "dataFilter": {"timeRange": {"from": OPTICAL_FROM, "to": OPTICAL_TO}},
                }
            ],
        },
        "output": {
            "width": src.width,
            "height": src.height,
            "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}],
        },
        "evalscript": EVALSCRIPT,
    }


work = []
for image in radar:
    with rasterio.open(image) as src:
        pu = estimate_pu(src.width, src.height, bands=3, float32=False, orthorectify=False)
        if not optical_path(image).is_file():
            work.append((image, optical_request(src), pu, src.profile))
cost = sum(w[2] for w in work)
print(
    f"{len(radar)} radar images, {len(work)} optical images to fetch, {cost:.1f} PU (budget {BUDGET_PU:.0f})"
)
assert cost <= BUDGET_PU, f"{cost:.1f} PU is over the budget of {BUDGET_PU}"

if work:
    OPTICAL_DIR.mkdir(parents=True, exist_ok=True)
    async with SentinelHub() as hub:
        for image, request, pu, profile in work:
            body = await hub.process(request, pu=pu)
            with rasterio.open(BytesIO(body)) as got:
                data = got.read()
            assert data.shape == (4, profile["height"], profile["width"]), data.shape
            profile = {**profile, "count": 4, "dtype": "uint16", "nodata": None}
            with rasterio.open(optical_path(image), "w", **profile) as dst:
                dst.write(data)
    print(f"spent about {hub.pu_estimated:.1f} PU")

## 5. Sentinel-2 under every chip

A pixel is **clear** where Sentinel-2 has data and SCL does not say cloud, cloud shadow, cirrus, snow or saturated. Among clear pixels, **water** is NDWI = (green − NIR) / (green + NIR) above a threshold, or SCL class 6.

In [ ]:
from rasterio.windows import Window

NOT_CLEAR = (0, 1, 3, 8, 9, 10, 11)
image_of = {(p.activation, p.product): p.image for ps in radar.values() for p in ps}


def optical_chip(row):
    res, size = float(row["resolution_m"]), int(row["size"])
    with rasterio.open(optical_path(image_of[row["activation"], row["product"]])) as src:
        col = round((float(row["x0"]) - src.transform.c) / res)
        line = round((src.transform.f - float(row["y0"])) / res)
        return src.read(window=Window(col, line, size, size), boundless=True, fill_value=0)


GREEN, NIR, SCL, HAS = (
    np.stack([optical_chip(rows[i]) for i in test]).transpose(1, 0, 2, 3).astype(np.float32)
)
CLEAR = (HAS > 0) & ~np.isin(SCL, NOT_CLEAR)
NDWI = (GREEN - NIR) / np.maximum(GREEN + NIR, 1)
print(f"clear on {CLEAR[OBSERVED[test]].mean():.1%} of the observed pixels of {TEST}")

## 6. What Sentinel-2 sees under each group of pixels

* **dark in both radar passes**: a control. It holds permanent water, but also shadow and tarmac.
* **mapped flood, found / missed**: the EMS polygons, split by the radar detector. The first row says how much water survives to the Sentinel-2 pass under a confirmed flood.
* **false alarms**: predicted by the radar detector, dry for EMS. This is the question.
* **quiet dry land**: dry for EMS, not predicted, a control for what NDWI marks with no flood.

Shares are of the clear pixels of each group.

In [ ]:
vv, pre = VV[test], PRE[test]
wet, dry = WET[test], OBSERVED[test] & ~WET[test]
predicted = PREDICTED[test]
GROUPS = {
    "dark in both radar passes": dry & (vv < t) & (pre < tp),
    "mapped flood, found": wet & predicted,
    "mapped flood, missed": wet & ~predicted,
    "false alarms": dry & predicted,
    "quiet dry land": dry & ~predicted & ~((vv < t) & (pre < tp)),
}


def table(chosen=None):
    results = {}
    for name, mask in GROUPS.items():
        mask = mask if chosen is None else mask & chosen
        clear = mask & CLEAR
        count = max(int(clear.sum()), 1)
        results[name] = {
            "km2": mask.sum() * km2,
            "clear": clear.sum() / max(int(mask.sum()), 1),
            "ndwi0": (clear & (NDWI > 0)).sum() / count,
            "ndwi2": (clear & (NDWI > 0.2)).sum() / count,
            "scl": (clear & (SCL == 6)).sum() / count,
        }
    return results


def show(results):
    print(f"{'':28} {'km²':>7} {'clear':>7} {'NDWI>0':>8} {'NDWI>0.2':>9} {'SCL water':>10}")
    for name, r in results.items():
        print(
            f"{name:28} {r['km2']:7.1f} {r['clear']:7.0%} {r['ndwi0']:8.1%} {r['ndwi2']:9.1%} {r['scl']:10.1%}"
        )


overall = table()
show(overall)

EMSR664 has two kinds of label, and AOI01 is mapped twice on one radar image, so the same table per product, with what each map was made from:

In [ ]:
products = sorted({rows[i]["product"] for i in test})
for product in products:
    chosen = np.array([rows[i]["product"] == product for i in test])[:, None, None]
    first = next(rows[i] for i in test if rows[i]["product"] == product)
    print(
        f"\n{product}, observed {first['observed_time'][:16]}, made from {first['made_from']}, gap {float(first['gap_hours']):+.1f} h"
    )
    show(table(chosen))

## 7. How much could the score change?

If every false alarm Sentinel-2 calls water were in fact flood, those pixels would become hits. The union of predicted and mapped water does not change, so the corrected CSI is (hits + confirmed) / (hits + misses + false alarms). It is a bound under an assumption, not a measured score: NDWI also marks some water that is not flood, which the quiet dry land row shows.

In [ ]:
false = GROUPS["false alarms"]
for name, water in {"NDWI > 0": NDWI > 0, "NDWI > 0.2": NDWI > 0.2, "SCL water": SCL == 6}.items():
    confirmed = int((false & CLEAR & water).sum())
    union = scores.hits + scores.misses + scores.false_alarms
    print(
        f"{name:12} {confirmed * km2:7.1f} km² of false alarms confirmed -> CSI {scores.csi:.3f} -> {(scores.hits + confirmed) / union:.3f}"
    )

## 8. Figure

Share of clear pixels that Sentinel-2 calls water (NDWI > 0), per group.

In [ ]:
import matplotlib.pyplot as plt

names = list(overall)
values = [overall[k]["ndwi0"] * 100 for k in names]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(names, values, height=0.6, color="#2a78d6")
for k, v in enumerate(values):
    ax.text(v + 1, k, f"{v:.1f} %", va="center", fontsize=9, color="#444")
ax.invert_yaxis()
ax.set_xlim(0, max(values) * 1.15 + 2)
ax.set_xlabel(f"clear pixels with NDWI > 0 on 23 May 2023, 10:06 UTC, {TEST} (%)")
ax.grid(axis="x", color="#e5e5e5")
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
fig.text(
    0.5,
    0.005,
    "Contains modified Copernicus Sentinel data and Copernicus EMS information, © European Union",
    ha="center",
    fontsize=7,
)
fig.tight_layout(rect=(0, 0.03, 1, 1))
FIGURE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE, dpi=110)
print("saved", FIGURE)

## Reading the scores

READING_PLACEHOLDER

*Data: Copernicus EMS Rapid Mapping, © European Union; Copernicus Sentinel-1 and Sentinel-2, processed by Sentinel Hub on the Copernicus Data Space Ecosystem.*